In [1]:
import multiprocessing
import numpy as np
import pandas as pd
import tensorflow_hub as hub
import tensorflow as tf
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument

In [2]:
# Embedding ~6M reviews in this dataset using SBERT, USE and Doc2Vec requires ~90-120min for each of them
df = pd.read_csv('data/dataset.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6417106 entries, 0 to 6417105
Data columns (total 5 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   app_id        int64 
 1   app_name      object
 2   review_text   object
 3   review_score  int64 
 4   review_votes  int64 
dtypes: int64(3), object(2)
memory usage: 244.8+ MB


In [4]:
def clean(texts):
    return texts.strip()

In [ ]:
''' 
SBERT can capture semantic relationships and support multilingual data,
it has high quality embedding but the embedding process may took time. 
The feature space of SBERT is 384 and has dense results.
'''

In [5]:
texts = df['review_text'].astype(str).apply(clean).tolist()
sbert = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

batch_size, l = 512, len(texts)
embedded_sbert = []
for i in tqdm(range(0, l, batch_size), desc='Embedding using SBERT'):
    batch = texts[i:min(l, i+batch_size)]
    emb_sbert = sbert.encode(
        batch,
        show_progress_bar=False,
        convert_to_numpy=True
    )
    embedded_sbert.append(emb_sbert)

Embedding using SBERT: 100%|███████████████████████████████████████████████████| 12534/12534 [1:30:06<00:00,  2.32it/s]


In [6]:
embedded_sbert = pd.DataFrame(np.vstack(embedded_sbert), columns=[f'emb{i}' for i in range(384)])
df_sbert = pd.concat([df.iloc[:, :3], embedded_sbert, df.iloc[:, 3:]], axis=1)

In [7]:
df_sbert.to_csv('results/df_sbert.csv', index=False)

In [ ]:
'''
Universal Sentence Encoder can capture semantic relationships and also support multilingual datasets like SBERT.
It has lower quality embeddings but faster speed than SBERT.
The feature space of it is 512.
'''

In [ ]:
texts = df['review_text'].astype(str).apply(clean).tolist()
use = hub.load('https://tfhub.dev/google/universal-sentence-encoder/4')

batch_size, l = 512, len(texts)
embedded_use = []
for i in tqdm(range(0, l, batch_size), desc='Embedding using USE'):
    batch = texts[i:min(l, i+batch_size)]
    emb_use = use(batch)
    embedded_use.append(emb_use)

In [ ]:
embedded_use = pd.DataFrame(np.vstack(embedded_use), columns=[f'emb{i}' for i in range(512)])
df_use = pd.concat([df.iloc[:, :3], embedded_use, df.iloc[:, 3:]], axis=1)

In [ ]:
df_use.to_csv('results/df_use.csv', index=False)

In [ ]:
'''
Doc2Vec need training on the Steam datasets (~6M reviews), which is good.
It captures some semantics relationship and weak in multilingual environments.
'''

In [ ]:
# There are ~6M Steam reviews, so it is better to apply simple cleaning in this case. The model can be more robust
def clean_doc2vec(texts):
    return texts.lower().strip()

def read_data(texts):
    for i, text in enumerate(texts):
        yield TaggedDocument(words=text.split(), tags=[str(i)])

In [ ]:
texts = df['review_text'].astype(str).apply(clean_doc2vec)

doc2vec = Doc2Vec(
    vector_size=200,
    window=5,
    min_count=5,
    workers=multiprocessing.cpu_count(),
    dm=0,
    epochs=1
)

doc2vec.build_vocab(read_data(texts))

In [ ]:
EPOCHS, l = 10, len(texts)
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    documents = read_data(texts)
    documents = tqdm(documents, total=l, desc="Docs")
    
    doc2vec.train(
        documents,
        total_examples=doc2vec.corpus_count,
        epochs=1
    )

In [ ]:
doc2vec.save('results/doc2vec_stream.model')

In [ ]:
doc_vectors = np.array([doc2vec.dv[str(i)] for i in range(l)])
embedded_doc2vec = pd.DataFrame(doc_vectors, columns=[f'emb{i}' for i in range(doc2vec.vector_size)])
df_doc2vec = pd.concat([df.iloc[:, :3], embedded_doc2vec, df.iloc[:, 3:]], axis=1)

In [ ]:
df_doc2vec.to_csv('results/df_doc2vec.csv', index=False)